# Solution F2 · Tree, logistic and KNN on four boundary shapes (and after a monotonic transformation)

**Type**: Compare

## Objective

Compare a tree (with `max_depth` and `min_samples_leaf` chosen in cross-validation), a logistic regression and a KNN on four boundary shapes in 2D (axis-parallel steps, XOR, diagonal, circle) and repeat the comparison after applying a **monotonic transformation** (`exp`) to the data: the tree does not change, the other two do.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


def shapes():
    rng = np.random.default_rng(0)
    n = 1500
    X = rng.uniform(-3, 3, (n, 2))
    labels = {
        "steps": ((X[:, 0] > 0.5) & (X[:, 1] < 1.0)) | ((X[:, 0] < -1.5) & (X[:, 1] > -0.5)),
        "XOR": (X[:, 0] > 0) ^ (X[:, 1] > 0),
        "diagonal": X[:, 0] + X[:, 1] > 0,
        "circle": (X ** 2).sum(1) < 4,
    }
    result = {}
    for name, y in labels.items():
        y = y.astype(int)
        y = np.where(rng.random(n) < 0.05, 1 - y, y)                    # 5% of wrong labels
        result[name] = (X, y)
    return result


data = shapes()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
grid = {"max_depth": [2, 3, 4, 5, 6, 8, None], "min_samples_leaf": [1, 5, 20]}

## Commented solution

In [ ]:
def evaluate(X_tr, X_te, y_tr, y_te, tree_params):
    tree = DecisionTreeClassifier(random_state=0, **tree_params).fit(X_tr, y_tr)
    logistic = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(15)).fit(X_tr, y_tr)
    return [tree.score(X_te, y_te), logistic.score(X_te, y_te), knn.score(X_te, y_te)]


table = pd.DataFrame(index=list(data), columns=["tree", "logistic", "knn"], dtype=float)
depths, best = {}, {}
for name, (X, y) in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=cv).fit(X_tr, y_tr)
    best[name], depths[name] = search.best_params_, search.best_params_["max_depth"]
    table.loc[name] = evaluate(X_tr, X_te, y_tr, y_te, search.best_params_)
print("chosen depths:", depths)
table.round(3)

In [ ]:
exp_table = pd.DataFrame(index=list(data), columns=["tree", "logistic", "knn"], dtype=float)
for name, (X, y) in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(np.exp(X), y, test_size=0.4, random_state=0)         # monotonic transformation of all the features
    exp_table.loc[name] = evaluate(X_tr, X_te, y_tr, y_te, best[name])
exp_table.round(3)

In [ ]:
change = exp_table - table
max_tree_change = float(change["tree"].abs().max())
max_other_change = float(change[["logistic", "knn"]].abs().max().max())
print(change.round(3))
print(f"maximum change: tree {max_tree_change:.3f} | logistic and KNN {max_other_change:.3f}")

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and set(depths) == set(data), "complete table and depth for each shape", "one row per shape")
    check((table.loc[["steps", "XOR", "circle"], "tree"] > table.loc[["steps", "XOR", "circle"], "logistic"] + 0.15).all() and table.loc["XOR", "logistic"] < 0.6, "the tree beats the logistic by over 15 points on steps, XOR and circle; on the XOR the logistic is at chance", "compare the columns")
    check(table.loc["diagonal", "logistic"] > table.loc["diagonal", "tree"] + 0.01 and table.loc["diagonal", "tree"] > 0.88, "on the diagonal the logistic beats the tree (which still goes over 0.88 with some depth)", "the diagonal is not parallel to the axes")
    check(depths["steps"] <= 5 and depths["diagonal"] is not None and depths["diagonal"] >= 4 and depths["circle"] >= depths["steps"], "chosen depth: the circle needs more depth than the steps; the diagonal needs at least 4", "best_params_['max_depth']")
    check(exp_table.notna().all().all(), "complete table with the exp transformation", "np.exp(X) for training and test")
    check(max_tree_change < 0.02 and max_other_change > 0.04, "after exp the tree changes by less than 2 points; logistic or KNN change by over 4 points", "change = exp_table - table")
    check(change.loc["diagonal", "logistic"] < -0.05, "the logistic on the diagonal gets worse by over 5 points with exp (the boundary is no longer a line)", "change['logistic']")
done()

## What you learned

- **The boundary decides the model.** With boundaries made of thresholds parallel to the axes (steps, XOR) the tree reaches 0.94-0.95 while the logistic is at 0.72 (steps) or at chance (XOR: no line separates the four quadrants). With the circle the tree gets 0.91 (it needs a staircase: depth 8 against 4-5 in the other cases).
  With the diagonal the logistic (0.94) beats the tree (0.91): describing an oblique line with right-angle thresholds takes many splits.
- **KNN is the generalist in 2D:** 0.93-0.94 on all the shapes, without having to guess the geometry (but remember the curse of dimensionality: in many dimensions it is no longer so, M5).
- **Invariance of trees:** after `exp` the tree changes by less than half a point (0.2 points on the steps shape: the few differences come from thresholds halfway between values that are now spaced differently), while logistic and KNN change by up to 11 points (the diagonal is no longer a line; the distances get distorted). Trees depend only on the **order** of the values:
  no scaling, no logarithms "to help", no worries about scale outliers.
- **A practical tip:** with tabular data the relationships are often "threshold-like" (age above 18, income above a value) and with interactions: it is the ground of trees (and of tree ensembles, modules 8 and 9). With physical measurements and smooth, diagonal relationships, linear models and networks are more efficient.

## Common pitfall

Concluding "trees are better than linear models" (or the opposite) from a single dataset. The answer depends on the geometry of the problem; and the first thing to do with new data is to try both.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When does a tree work better than a logistic regression?</summary>

When the relationship with the target is made of thresholds and interactions (effects that exist only for certain ranges of several variables) rather than smooth linear combinations: XOR, "if A and not B" rules, step effects. With oblique boundaries or many weakly informative features the regularized logistic is often better
and more stable. Another advantage of trees is the invariance to monotonic transformations of the features: no scaling. I compare them in cross-validation on the same metric.

</details>